# Single Object Detection — Model Serving (Colab + ngrok)

**Bootcamp**: rubythalib.ai — mentor Daniel Syahputra

⚠️ **Kompatibilitas arsitektur**: notebook ini pakai arsitektur **v3** (split head: `objectness_head` + `bbox_head`). Kalau `best.pt` yang kamu load berasal dari training **v1/v2** (head gabungan `classifier`), `load_state_dict` akan gagal dengan error "Missing key(s)/Unexpected key(s)". Pastikan versi notebook training & serving yang kamu pakai **sama-sama v3**.

Versi serving ini dipakai kalau kamu deploy langsung dari Colab (tanpa server sendiri). Beda dari `app.py` versi lokal:
1. **Tunneling** via `pyngrok` — endpoint bisa diakses dari luar Colab
2. **Visualisasi** — bounding box digambar langsung di atas gambar pakai `cv2.rectangle`
3. **Response** — gambar hasil anotasi dikirim balik sebagai `base64` di JSON

Checkpoint (`best.pt`) diambil otomatis dari Google Drive — path yang sama dengan tempat notebook training menyimpannya. Tidak perlu upload manual lagi.


In [ ]:
!pip install fastapi uvicorn pyngrok torch torchvision Pillow opencv-python-headless python-multipart -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Path ini HARUS sama dengan DRIVE_CHECKPOINT_DIR di notebook training
CHECKPOINT_PATH = '/content/drive/MyDrive/single_object_detection/checkpoints/best.pt'


In [ ]:
import base64
import io
import cv2
import numpy as np
import torch
import threading
import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
from PIL import Image
from torch import nn
from torchvision import transforms
from torchvision.models import mobilenet_v2
from pyngrok import ngrok


In [ ]:
def cv2_image_to_base64(image):
    _, buffer = cv2.imencode(".jpg", image)
    return base64.b64encode(buffer).decode("utf-8")


class ObjectDetectionModel(nn.Module):
    def __init__(self, freeze_until: int = 14):
        super(ObjectDetectionModel, self).__init__()
        self.backbone = mobilenet_v2(weights="DEFAULT").features

        for i, block in enumerate(self.backbone):
            if i < freeze_until:
                for param in block.parameters():
                    param.requires_grad = False

        # HARUS identik dengan arsitektur di notebook training (v3, split head) --
        # kalau training diganti lagi, update juga di sini, atau load_state_dict akan gagal.
        self.objectness_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

        self.bbox_head = nn.Sequential(
            nn.Conv2d(1280, 256, kernel_size=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 4),
        )

    def forward(self, x):
        features = self.backbone(x)
        is_object = torch.sigmoid(self.objectness_head(features))
        bbox = torch.sigmoid(self.bbox_head(features))
        return is_object, bbox


In [ ]:
app = FastAPI(title="Single Object Detection API (Colab)")
model = ObjectDetectionModel()
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=torch.device("cpu")))
model.eval()
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])


@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    try:
        image = Image.open(io.BytesIO(await file.read())).convert("RGB")
        cv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        img_width, img_height = image.size
        image_tensor = transform(image).unsqueeze(0)

        with torch.no_grad():
            is_object, bbox = model(image_tensor)

        is_object_value = is_object.item()
        x, y, w, h = bbox.squeeze(0).tolist()
        x, y, w, h = x * img_width, y * img_height, w * img_width, h * img_height
        x1, y1, x2, y2 = int(x - w / 2), int(y - h / 2), int(x + w / 2), int(y + h / 2)
        cv2.rectangle(cv_image, (x1, y1), (x2, y2), (0, 255, 0), 1)  # gambar bbox di atas image

        if is_object_value > 0.5:
            response = {
                "is_object": True,
                "bbox": {"x": x, "y": y, "w": w, "h": h},
                "image_base64": cv2_image_to_base64(cv_image),  # kirim image hasil anotasi
            }
        else:
            response = {"is_object": False, "bbox": None}
        return JSONResponse(content=response)
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})


@app.get("/")
def root():
    return {"message": "Single Object Detection API is running!"}


## Jalankan server + tunnel ngrok

Ganti `GANTI_DENGAN_TOKEN_NGROK_ANDA` dengan token asli dari [dashboard.ngrok.com](https://dashboard.ngrok.com) sebelum menjalankan cell ini.

⚠️ **Jangan pernah commit token ngrok ke GitHub** — kalau ini notebook yang mau di-push, pastikan cell ini tidak menyimpan token asli di history/output.

In [ ]:
ngrok.set_auth_token("GANTI_DENGAN_TOKEN_NGROK_ANDA")   # ambil token di dashboard.ngrok.com
nest_asyncio.apply()

def run_app():
    uvicorn.run(app, host="0.0.0.0", port=8000)

ngrok_tunnel = ngrok.connect(8000)
print(f"Public URL: {ngrok_tunnel.public_url}")
threading.Thread(target=run_app).start()


## Cara test

Setelah server jalan, kirim request dari terminal/Postman/browser lain:

```bash
curl -X POST "<PUBLIC_URL_DARI_NGROK>/predict" -F "file=@contoh_gambar.jpg"
```

Response akan berisi `is_object`, `bbox` (koordinat piksel), dan `image_base64` (gambar dengan bounding box tergambar) kalau objek terdeteksi.